<a href="https://colab.research.google.com/github/LCaravaggio/Happiness_Polarization/blob/main/Protestas_ACLED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El problema con ACLED es que solo tiene datos de 2019 en adelante

In [23]:
from google.colab import userdata

ACLED_TOKEN = userdata.get('ACLED')

In [24]:
import requests
import pandas as pd

url = "https://acleddata.com/api/acled/read?_format=json"

def get_access_token(username, password, token_url):
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
    }
    data = {
          'username': username,
          'password': password,
          'grant_type': "password",
          'client_id': "acled"
    }

    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code == 200:
        token_data = response.json()
        return token_data['access_token']
    else:
        raise Exception(f"Failed to get access token: {response.status_code} {response.text}")

my_token = get_access_token(
    username="lcaravaggio@itba.edu.ar",
    password=ACLED_TOKEN,
    token_url="https://acleddata.com/oauth/token",
)

url = "https://acleddata.com/api/acled/read?_format=json"

In [29]:
paises = (
    "Argentina|Bolivia|Brazil|Chile|Colombia|Costa Rica|"
    "Dominican Republic|Ecuador|El Salvador|Guatemala|Honduras|"
    "Mexico|Nicaragua|Panama|Paraguay|Peru|Uruguay|Venezuela"
)

params = {
    "country": paises,
    "year": "2000|2024",
    "year_where": "BETWEEN",
    "tags": "crowd size=large|crowd size=massive",

    "fields": (
        "event_id_cnty|event_date|year|time_precision|"
        "country|admin1|admin2|location|"
        "event_type|sub_event_type|fatalities|notes|tags"
    )
}

response = requests.get(
    url,
    params=params,
    headers={
        "Authorization": f"Bearer {my_token}",
        "Content-Type": "application/json"
    }
)

response.raise_for_status()

data = response.json()

protestas_importantes = pd.DataFrame(data["data"])

protestas_importantes["event_date"] = pd.to_datetime(
    protestas_importantes["event_date"],
    errors="coerce"
)

protestas_primera = (
    protestas_importantes
    .sort_values(['country', 'event_date'])
    .drop_duplicates(
        subset=['country', 'year'],
        keep='first'
    )
    .reset_index(drop=True)
)
protestas_primera.to_csv('protestas.csv')

In [30]:
len(protestas_primera)

102